# 03 — WT | Statistics, summaries and behavioral heatmaps

**Fast analysis only.** Reads precomputed `04_results/WT/electrophysiology_v1/WT_mouse_region_phase_metrics.csv`. Never accesses raw Neuropixels probes.

Current validated example: DG firing rate pairwise heatmap (column minus row; Holm-adjusted stars). Additional region-metric tests are exploratory. The full analysis has six states (quiescence, stimulus, movement, reward, error, ITI).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import pandas as pd, numpy as np, matplotlib.pyplot as plt
ROOT = Path('/content/drive/MyDrive/Reanalysis/Our_Analysis/Behavior_aligned')
DATA = ROOT / '04_results' / 'WT' / 'electrophysiology_v1'
assert (DATA / 'WT_mouse_region_phase_metrics.csv').exists(), f'Missing processed file: {DATA}'
print('Found mouse-level data:', DATA)


## Firing-rate test: 9 regions

In [ ]:

# ============================================================
# STEP 6 — WT FIRING RATE: REPEATED-MEASURES STATISTICS
# Friedman omnibus + paired Wilcoxon + Holm correction
# Independent biological unit: mouse
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import friedmanchisquare, wilcoxon
from statsmodels.stats.multitest import multipletests
from itertools import combinations
from IPython.display import display

ROOT = Path("/content/drive/MyDrive/Reanalysis/Our_Analysis/Behavior_aligned")

DATA = (
    ROOT / "04_results" / "WT"
    / "electrophysiology_v1"
)

STATES = ["Q", "S", "M", "F_reward", "F_error", "I"]

REGIONS = [
    "DG", "CA1", "CA3", "SUB", "VISp",
    "MOs", "CP", "MRN", "APN"
]

MIN_MICE = 5

df = pd.read_csv(
    DATA / "WT_mouse_region_phase_metrics.csv"
)

metric = "fr_hz"

omnibus_results = []
posthoc_results = []

for region in REGIONS:

    sub = df.loc[df["region"] == region]

    matrix = sub.pivot_table(
        index="subject",
        columns="state",
        values=metric,
        aggfunc="mean"
    ).reindex(columns=STATES)

    # ------------------------------------------
    # FRIEDMAN — complete repeated observations
    # ------------------------------------------

    complete = matrix.dropna()

    if len(complete) >= MIN_MICE:

        stat, p_friedman = friedmanchisquare(
            *[complete[s].to_numpy() for s in STATES]
        )

        n = len(complete)
        k = len(STATES)

        kendall_w = stat / (n * (k - 1))

    else:

        n = len(complete)
        stat = np.nan
        p_friedman = np.nan
        kendall_w = np.nan

    omnibus_results.append({
        "region": region,
        "n_complete_mice": n,
        "friedman_chi2": stat,
        "p_friedman": p_friedman,
        "kendall_W": kendall_w
    })

    # ------------------------------------------
    # PAIRED WILCOXON — all state comparisons
    # ------------------------------------------

    local_results = []

    for state_a, state_b in combinations(STATES, 2):

        paired = matrix[
            [state_a, state_b]
        ].dropna()

        n_pair = len(paired)

        mean_a = paired[state_a].mean()
        mean_b = paired[state_b].mean()

        difference = (
            paired[state_b] - paired[state_a]
        )

        if n_pair >= MIN_MICE:

            if np.allclose(difference.to_numpy(), 0):
                p_raw = 1.0
                statistic = 0.0
            else:
                result = wilcoxon(
                    paired[state_a],
                    paired[state_b],
                    alternative="two-sided",
                    zero_method="wilcox",
                    method="auto"
                )

                p_raw = result.pvalue
                statistic = result.statistic

        else:
            p_raw = np.nan
            statistic = np.nan

        local_results.append({
            "region": region,
            "state_A": state_a,
            "state_B": state_b,
            "n_mice": n_pair,
            "mean_A_Hz": mean_a,
            "mean_B_Hz": mean_b,
            "mean_difference_Hz": difference.mean(),
            "median_paired_difference_Hz": difference.median(),
            "wilcoxon_W": statistic,
            "p_raw": p_raw
        })

    # Holm correction across the valid comparisons
    # within this anatomical region.
    valid_idx = [
        i for i, row in enumerate(local_results)
        if np.isfinite(row["p_raw"])
    ]

    if valid_idx:

        raw_p = [
            local_results[i]["p_raw"]
            for i in valid_idx
        ]

        _, adjusted, _, _ = multipletests(
            raw_p,
            alpha=0.05,
            method="holm"
        )

        for i, p_adj in zip(valid_idx, adjusted):
            local_results[i]["p_holm"] = p_adj

    posthoc_results.extend(local_results)


# ============================================================
# SAVE RESULTS
# ============================================================

omnibus = pd.DataFrame(omnibus_results)
posthoc = pd.DataFrame(posthoc_results)

# Also adjust the 9 omnibus tests across regions.
valid = omnibus["p_friedman"].notna()

omnibus["p_fdr_9_regions"] = np.nan

if valid.any():
    omnibus.loc[valid, "p_fdr_9_regions"] = (
        multipletests(
            omnibus.loc[valid, "p_friedman"],
            method="fdr_bh"
        )[1]
    )

posthoc["significant_raw"] = posthoc["p_raw"] < 0.05
posthoc["significant_holm"] = posthoc["p_holm"] < 0.05

omnibus.to_csv(
    DATA / "WT_FR_Friedman_all_regions.csv",
    index=False
)

posthoc.to_csv(
    DATA / "WT_FR_Wilcoxon_all_pairs.csv",
    index=False
)

# ============================================================
# DISPLAY
# ============================================================

print("=" * 65)
print("FRIEDMAN — FIRING RATE ACROSS BEHAVIORAL STATES")
print("=" * 65)

display(
    omnibus.sort_values("p_friedman").round(5)
)

print("\n" + "=" * 65)
print("SIGNIFICANT PAIRWISE COMPARISONS — HOLM CORRECTED")
print("=" * 65)

significant = posthoc.loc[
    posthoc["significant_holm"]
].sort_values("p_holm")

if len(significant):

    display(significant.round(5))

else:

    print("No Holm-corrected significant pairs.")

print("\n" + "=" * 65)
print("NOMINALLY SIGNIFICANT PAIRS — RAW p < 0.05")
print("=" * 65)

nominal = posthoc.loc[
    posthoc["significant_raw"]
].sort_values("p_raw")

display(nominal.round(5))

print("\nSUMMARY")
print("Omnibus p < 0.05:",
      int((omnibus["p_friedman"] < 0.05).sum()))

print("Omnibus FDR < 0.05:",
      int((omnibus["p_fdr_9_regions"] < 0.05).sum()))

print("Pairwise raw p < 0.05:",
      len(nominal))

print("Pairwise Holm p < 0.05:",
      len(significant))

print("\nResults saved to:", DATA)


## All 10 metrics: 9-region comparisons

In [ ]:

# ============================================================
# STEP 7 — COMPLETE WT BEHAVIORAL-STATE COMPARISON
#
# 9 brain regions x 10 electrophysiological metrics
# 6 behavioral states
#
# Friedman repeated-measures test
# Paired Wilcoxon post-hoc with Holm correction
# BH-FDR correction across all omnibus tests
#
# Outputs: statistics CSV + 12 publication-style figures
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from pathlib import Path
from itertools import combinations
from scipy.stats import friedmanchisquare, wilcoxon
from statsmodels.stats.multitest import multipletests
from IPython.display import display

ROOT = Path("/content/drive/MyDrive/Reanalysis/Our_Analysis/Behavior_aligned")

DATA = (
    ROOT / "04_results" / "WT" / "electrophysiology_v1"
)

OUT = DATA / "All_metrics_state_comparison"

FIGURES = (
    ROOT / "05_figures" / "WT" / "All_metrics_state_comparison"
)

OUT.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

MIN_MICE = 5
ALPHA = 0.05

REGIONS = [
    "DG", "CA1", "CA3", "SUB",
    "VISp", "MOs", "CP", "MRN", "APN"
]

STATES = [
    "Q", "S", "M",
    "F_reward", "F_error", "I"
]

STATE_LABELS = [
    "Q", "S", "M", "Reward", "Error", "ITI"
]

STATE_COLORS = [
    "#8298C6",
    "#61AD94",
    "#E39A5B",
    "#B38AC7",
    "#D47C89",
    "#899CAA"
]

METRICS = {
    "fr_hz": ("Firing rate", "Hz"),
    "mean_isi_ms": ("Mean ISI", "ms"),
    "cv_isi": ("CV ISI", ""),
    "cv2": ("CV2", ""),
    "burst_rate_min": ("Burst rate", "bursts/min"),
    "burst_fraction_pct": ("Burst fraction", "%"),
    "burst_duration_ms": ("Burst duration", "ms"),
    "spikes_per_burst": ("Spikes per burst", "spikes"),
    "intraburst_hz": ("Intraburst frequency", "Hz"),
    "interburst_interval_s": ("Interburst interval", "s")
}

# ------------------------------------------------------------
# 1. LOAD PREVIOUSLY CALCULATED MOUSE-LEVEL VALUES
# ------------------------------------------------------------

df = pd.read_csv(
    DATA / "WT_mouse_region_phase_metrics.csv"
)

missing = [
    m for m in METRICS
    if m not in df.columns
]

if missing:
    raise ValueError(f"Missing metrics: {missing}")

print("WT mice:", df["subject"].nunique())
print("Regions:", df["region"].nunique())
print("Metrics:", len(METRICS))
print("States:", len(STATES))

# ------------------------------------------------------------
# 2. RUN FRIEDMAN + WILCOXON FOR EVERY REGION/METRIC
# ------------------------------------------------------------

omnibus_rows = []
posthoc_rows = []

# Keep paired data for all figures.
matrices = {}

for region in REGIONS:

    subset = df.loc[df["region"] == region]

    for metric in METRICS:

        matrix = subset.pivot_table(
            index="subject",
            columns="state",
            values=metric,
            aggfunc="mean"
        ).reindex(columns=STATES)

        # The same mice are used across all six states.
        complete = matrix.dropna()

        matrices[(region, metric)] = complete

        n = len(complete)
        k = len(STATES)

        chi2 = np.nan
        p = np.nan
        kendall_w = np.nan

        if n >= MIN_MICE:

            values = [
                complete[s].to_numpy(float)
                for s in STATES
            ]

            # Friedman is undefined when all states are
            # numerically identical for all mice.
            try:
                if np.allclose(
                    np.column_stack(values),
                    np.column_stack(values)[:, :1]
                ):
                    chi2, p, kendall_w = 0.0, 1.0, 0.0
                else:
                    chi2, p = friedmanchisquare(*values)
                    kendall_w = chi2 / (n * (k - 1))

                    if not np.isfinite(p):
                        chi2, p, kendall_w = 0.0, 1.0, 0.0

            except ValueError:
                chi2, p, kendall_w = np.nan, np.nan, np.nan

        omnibus_rows.append({
            "region": region,
            "metric": metric,
            "n_mice": n,
            "friedman_chi2": chi2,
            "p_raw": p,
            "kendall_W": kendall_w
        })

        local = []

        for a, b in combinations(STATES, 2):

            pair = complete[[a, b]]
            diff = pair[b] - pair[a]

            p_pair = np.nan
            w_stat = np.nan

            if n >= MIN_MICE:

                if np.allclose(diff.to_numpy(float), 0):
                    p_pair = 1.0
                    w_stat = 0.0

                else:
                    try:
                        result = wilcoxon(
                            pair[a],
                            pair[b],
                            alternative="two-sided",
                            zero_method="wilcox",
                            method="auto"
                        )

                        p_pair = result.pvalue
                        w_stat = result.statistic

                    except ValueError:
                        pass

            local.append({
                "region": region,
                "metric": metric,
                "state_A": a,
                "state_B": b,
                "n_mice": n,
                "mean_A": pair[a].mean(),
                "mean_B": pair[b].mean(),
                "mean_difference_B_minus_A": diff.mean(),
                "median_paired_difference": diff.median(),
                "wilcoxon_W": w_stat,
                "p_raw": p_pair
            })

        # Holm correction within each region x metric.
        valid_idx = [
            i for i, row in enumerate(local)
            if np.isfinite(row["p_raw"])
        ]

        if valid_idx:

            adjusted = multipletests(
                [local[i]["p_raw"] for i in valid_idx],
                method="holm"
            )[1]

            for i, adj in zip(valid_idx, adjusted):
                local[i]["p_holm"] = adj

        posthoc_rows.extend(local)


omnibus = pd.DataFrame(omnibus_rows)
posthoc = pd.DataFrame(posthoc_rows)

# ------------------------------------------------------------
# 3. MULTIPLE-TEST CORRECTION ACROSS ALL 90 OMNIBUS TESTS
# ------------------------------------------------------------

omnibus["p_fdr"] = np.nan

valid = omnibus["p_raw"].notna()

if valid.any():

    omnibus.loc[valid, "p_fdr"] = multipletests(
        omnibus.loc[valid, "p_raw"],
        method="fdr_bh"
    )[1]

omnibus["significant_raw"] = omnibus["p_raw"] < ALPHA
omnibus["significant_fdr"] = omnibus["p_fdr"] < ALPHA

posthoc["p_holm"] = pd.to_numeric(
    posthoc.get("p_holm"),
    errors="coerce"
)

posthoc["significant_raw"] = posthoc["p_raw"] < ALPHA
posthoc["significant_holm"] = posthoc["p_holm"] < ALPHA

# Save all statistics.
omnibus.to_csv(
    OUT / "WT_ALL_metrics_Friedman.csv",
    index=False
)

posthoc.to_csv(
    OUT / "WT_ALL_metrics_Wilcoxon_posthoc.csv",
    index=False
)

omnibus.loc[
    omnibus["significant_fdr"]
].to_csv(
    OUT / "WT_significant_omnibus_FDR.csv",
    index=False
)

posthoc.loc[
    posthoc["significant_holm"]
].to_csv(
    OUT / "WT_significant_posthoc_Holm.csv",
    index=False
)

print("\n" + "=" * 65)
print("STATISTICAL SUMMARY")
print("=" * 65)

print("Omnibus tests:", valid.sum())
print("Friedman raw p < 0.05:",
      int(omnibus["significant_raw"].sum()))
print("Friedman FDR p < 0.05:",
      int(omnibus["significant_fdr"].sum()))
print("Wilcoxon raw p < 0.05:",
      int(posthoc["significant_raw"].sum()))
print("Wilcoxon Holm p < 0.05:",
      int(posthoc["significant_holm"].sum()))

print("\nSIGNIFICANT OMNIBUS RESULTS")
display(
    omnibus.loc[
        omnibus["significant_fdr"]
    ].sort_values("p_fdr").round(5)
)

print("\nSIGNIFICANT POST-HOC RESULTS")
display(
    posthoc.loc[
        posthoc["significant_holm"]
    ].sort_values("p_holm").round(5)
)

# ------------------------------------------------------------
# 4. DRAW ONE 3x3 FIGURE PER ELECTROPHYSIOLOGICAL METRIC
# ------------------------------------------------------------

def stars(p):
    if not np.isfinite(p):
        return ""
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return ""


for metric, (metric_name, unit) in METRICS.items():

    fig, axes = plt.subplots(
        3, 3,
        figsize=(18, 14),
        sharex=True
    )

    # Reserve space for title and explanatory footer.
    fig.subplots_adjust(
        top=0.91,
        bottom=0.10,
        left=0.07,
        right=0.98,
        hspace=0.42,
        wspace=0.24
    )

    for ax, region in zip(axes.flat, REGIONS):

        matrix = matrices[(region, metric)]
        n = len(matrix)

        if n < MIN_MICE:

            ax.text(
                0.5, 0.5,
                f"Insufficient mice\nn = {n}",
                transform=ax.transAxes,
                ha="center", va="center",
                fontsize=11,
                color="#777777"
            )
            ax.set_title(region, fontweight="bold")
            continue

        values = matrix[STATES].to_numpy(float)

        # Individual mouse trajectories.
        for mouse_values in values:

            ax.plot(
                range(len(STATES)),
                mouse_values,
                color="#A9A9A9",
                linewidth=0.8,
                alpha=0.28,
                zorder=1
            )

        # Mean +/- SEM and mouse observations.
        for j, state in enumerate(STATES):

            y = matrix[state].to_numpy(float)

            mean = y.mean()
            sem = y.std(ddof=1) / np.sqrt(n)

            ax.scatter(
                np.full(n, j),
                y,
                s=17,
                color=STATE_COLORS[j],
                alpha=0.55,
                zorder=2
            )

            ax.errorbar(
                j, mean, yerr=sem,
                fmt="o",
                color="#202020",
                markerfacecolor="white",
                markeredgewidth=1.5,
                markersize=7,
                capsize=3.5,
                linewidth=1.5,
                zorder=5
            )

        # FDR-adjusted omnibus significance in title.
        row = omnibus.loc[
            (omnibus["region"] == region) &
            (omnibus["metric"] == metric)
        ].iloc[0]

        mark = stars(row["p_fdr"])

        title = (
            f"{region}  {mark}  "
            f"(n={n}, FDR p={row['p_fdr']:.3g})"
        )

        ax.set_title(
            title,
            fontsize=11,
            fontweight="bold"
        )

        # Compact summary of significant Holm comparisons.
        sig_pairs = posthoc.loc[
            (posthoc["region"] == region) &
            (posthoc["metric"] == metric) &
            (posthoc["significant_holm"])
        ].sort_values("p_holm")

        if len(sig_pairs):

            labels = []

            for _, pair in sig_pairs.head(3).iterrows():

                labels.append(
                    f"{pair['state_A']}–{pair['state_B']} "
                    f"{stars(pair['p_holm'])}"
                )

            annotation = "  |  ".join(labels)

            if len(sig_pairs) > 3:
                annotation += f"  (+{len(sig_pairs)-3} more)"

            ax.text(
                0.5, 0.99,
                annotation,
                transform=ax.transAxes,
                ha="center", va="top",
                fontsize=8,
                color="#9B3B45",
                bbox={
                    "facecolor": "white",
                    "edgecolor": "none",
                    "alpha": 0.88,
                    "pad": 2
                }
            )

        ax.set_ylabel(
            f"{metric_name} ({unit})" if unit
            else metric_name,
            fontsize=9
        )

        ax.set_xticks(range(len(STATES)))
        ax.set_xticklabels(
            STATE_LABELS,
            rotation=35,
            ha="right",
            fontsize=9
        )

        ax.grid(axis="y", alpha=0.15)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    fig.suptitle(
        f"WT — {metric_name} across behavioral states",
        fontsize=18,
        fontweight="bold"
    )

    fig.text(
        0.5, 0.025,
        "Mouse-level paired observations | "
        "Open circles = mean ± SEM | "
        "Title stars = Friedman BH-FDR | "
        "Pair labels = Wilcoxon Holm",
        ha="center",
        fontsize=10,
        color="#555555"
    )

    file = FIGURES / f"WT_{metric}_all_regions"

    fig.savefig(
        file.with_suffix(".png"),
        dpi=300,
        bbox_inches="tight"
    )

    fig.savefig(
        file.with_suffix(".pdf"),
        bbox_inches="tight"
    )

    plt.close(fig)

    print("Saved:", metric_name)


# ------------------------------------------------------------
# 5. OVERVIEW HEATMAP: FRIEDMAN SIGNIFICANCE
# ------------------------------------------------------------

p_matrix = (
    omnibus.pivot(
        index="region",
        columns="metric",
        values="p_fdr"
    )
    .reindex(index=REGIONS, columns=list(METRICS))
)

# Use -log10(FDR p), capped for display only.
heat_data = -np.log10(
    p_matrix.clip(lower=1e-12).to_numpy(float)
)

fig, ax = plt.subplots(figsize=(17, 6.5))

im = ax.imshow(
    np.ma.masked_invalid(heat_data),
    aspect="auto",
    cmap="Blues",
    vmin=0,
    vmax=4
)

for i in range(len(REGIONS)):
    for j in range(len(METRICS)):

        p = p_matrix.iloc[i, j]

        if np.isfinite(p):
            label = stars(p)
        else:
            label = "NA"

        ax.text(
            j, i, label,
            ha="center",
            va="center",
            fontsize=11,
            color="black",
            fontweight="bold"
        )

ax.set_xticks(range(len(METRICS)))
ax.set_xticklabels(
    [METRICS[m][0] for m in METRICS],
    rotation=45,
    ha="right"
)

ax.set_yticks(range(len(REGIONS)))
ax.set_yticklabels(REGIONS)

ax.set_title(
    "WT — Behavioral-state effects across brain regions\n"
    "Friedman tests, BH-FDR corrected",
    fontsize=15,
    fontweight="bold"
)

cbar = fig.colorbar(im, ax=ax, shrink=0.85)
cbar.set_label("-log10(FDR-adjusted p)")

fig.tight_layout()

fig.savefig(
    FIGURES / "WT_omnibus_significance_heatmap.png",
    dpi=300,
    bbox_inches="tight"
)

fig.savefig(
    FIGURES / "WT_omnibus_significance_heatmap.pdf",
    bbox_inches="tight"
)

plt.show()
plt.close(fig)


# ------------------------------------------------------------
# 6. OVERVIEW HEATMAP: NUMBER OF SIGNIFICANT PAIRS
# ------------------------------------------------------------

pair_counts = (
    posthoc.groupby(["region", "metric"])[
        "significant_holm"
    ]
    .sum()
    .unstack("metric")
    .reindex(
        index=REGIONS,
        columns=list(METRICS)
    )
    .fillna(0)
    .astype(int)
)

fig, ax = plt.subplots(figsize=(17, 6.5))

im = ax.imshow(
    pair_counts.to_numpy(),
    aspect="auto",
    cmap="YlOrRd",
    vmin=0,
    vmax=15
)

for i in range(len(REGIONS)):
    for j in range(len(METRICS)):

        count = pair_counts.iloc[i, j]

        ax.text(
            j, i, str(count),
            ha="center",
            va="center",
            fontsize=11,
            fontweight="bold"
        )

ax.set_xticks(range(len(METRICS)))
ax.set_xticklabels(
    [METRICS[m][0] for m in METRICS],
    rotation=45,
    ha="right"
)

ax.set_yticks(range(len(REGIONS)))
ax.set_yticklabels(REGIONS)

ax.set_title(
    "WT — Significant behavioral-state comparisons\n"
    "Number of Wilcoxon pairs surviving Holm correction",
    fontsize=15,
    fontweight="bold"
)

cbar = fig.colorbar(im, ax=ax, shrink=0.85)
cbar.set_label("Significant pairs (out of 15)")

fig.tight_layout()

fig.savefig(
    FIGURES / "WT_significant_posthoc_count_heatmap.png",
    dpi=300,
    bbox_inches="tight"
)

fig.savefig(
    FIGURES / "WT_significant_posthoc_count_heatmap.pdf",
    bbox_inches="tight"
)

plt.show()
plt.close(fig)


# ------------------------------------------------------------
# 7. EXPORT COMPACT SUMMARY TABLE
# ------------------------------------------------------------

summary = omnibus.merge(
    pair_counts.stack().rename(
        "n_significant_holm_pairs"
    ).reset_index(),
    on=["region", "metric"],
    how="left"
)

summary.to_csv(
    OUT / "WT_ALL_metrics_statistical_overview.csv",
    index=False
)

print("\n" + "=" * 70)
print("COMPLETE WT BEHAVIORAL-STATE ANALYSIS FINISHED")
print("=" * 70)

print("Regions:", len(REGIONS))
print("Electrophysiological metrics:", len(METRICS))
print("Total region-metric combinations:", len(omnibus))
print("FDR-significant omnibus:", int(omnibus["significant_fdr"].sum()))
print("Holm-significant paired comparisons:",
      int(posthoc["significant_holm"].sum()))
print("Figures: 12 PNG + 12 PDF")
print("Statistics:", OUT)
print("Figures:", FIGURES)

display(
    summary.sort_values("p_fdr").head(20).round(5)
)


## Pilot 6×6 heatmap: DG firing rate

In [ ]:

# ============================================================
# PILOT HEATMAP
# DG — FIRING RATE ACROSS BEHAVIORAL STATES
#
# Cell = column state MINUS row state, in Hz.
# Red: column has higher firing rate.
# Blue: column has lower firing rate.
# Stars: paired Wilcoxon, Holm-corrected p.
#
# Biological unit: mouse
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from itertools import combinations
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests
from matplotlib.colors import TwoSlopeNorm
from IPython.display import display

ROOT = Path("/content/drive/MyDrive/Reanalysis/Our_Analysis/Behavior_aligned")

DATA = (
    ROOT / "04_results" / "WT" / "electrophysiology_v1"
)

FIGURES = (
    ROOT / "05_figures" / "WT" / "Pilot_heatmaps"
)

FIGURES.mkdir(parents=True, exist_ok=True)

REGION = "DG"
METRIC = "fr_hz"
MIN_MICE = 5

STATES = [
    "Q", "S", "M", "F_reward", "F_error", "I"
]

LABELS = [
    "Quiescence",
    "Stimulus",
    "Movement",
    "Reward",
    "Error",
    "Inter-trial"
]

# ------------------------------------------------------------
# 1. LOAD MOUSE-LEVEL RESULTS
# ------------------------------------------------------------

df = pd.read_csv(
    DATA / "WT_mouse_region_phase_metrics.csv"
)

sub = df.loc[df["region"] == REGION]

matrix = sub.pivot_table(
    index="subject",
    columns="state",
    values=METRIC,
    aggfunc="mean"
).reindex(columns=STATES)

# Use the same mice across all six states.
matrix = matrix.dropna()

n_mice = len(matrix)

if n_mice < MIN_MICE:
    raise RuntimeError(
        f"Insufficient mice: {n_mice}"
    )

print(f"Region: {REGION}")
print(f"Metric: {METRIC}")
print(f"Complete paired mice: {n_mice}")

print("\nMEAN FIRING RATE BY STATE (Hz)")
display(
    matrix.mean().rename("Mean FR (Hz)").round(3)
)

# ------------------------------------------------------------
# 2. PAIRED DIFFERENCES AND WILCOXON TESTS
# ------------------------------------------------------------

pair_results = []

for state_a, state_b in combinations(STATES, 2):

    paired_diff = (
        matrix[state_b] - matrix[state_a]
    )

    if np.allclose(paired_diff.to_numpy(), 0):
        stat, p_raw = 0.0, 1.0
    else:
        stat, p_raw = wilcoxon(
            matrix[state_a],
            matrix[state_b],
            alternative="two-sided",
            zero_method="wilcox",
            method="auto"
        )

    pair_results.append({
        "row_state": state_a,
        "column_state": state_b,
        "n_mice": n_mice,
        "mean_difference_hz": paired_diff.mean(),
        "median_difference_hz": paired_diff.median(),
        "p_raw": p_raw,
        "wilcoxon_W": stat
    })

results = pd.DataFrame(pair_results)

results["p_holm"] = multipletests(
    results["p_raw"],
    method="holm"
)[1]

# ------------------------------------------------------------
# 3. BUILD SYMMETRIC COMPARISON MATRICES
# ------------------------------------------------------------

k = len(STATES)

differences = np.zeros((k, k))
p_values = np.full((k, k), np.nan)

for _, row in results.iterrows():

    i = STATES.index(row["row_state"])
    j = STATES.index(row["column_state"])

    d = float(row["mean_difference_hz"])
    p = float(row["p_holm"])

    differences[i, j] = d
    differences[j, i] = -d

    p_values[i, j] = p
    p_values[j, i] = p


def significance_stars(p):
    if not np.isfinite(p):
        return ""
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return ""


# ------------------------------------------------------------
# 4. DRAW HEATMAP
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(10.5, 8.5))

max_abs = np.max(np.abs(differences))

norm = TwoSlopeNorm(
    vmin=-max_abs,
    vcenter=0,
    vmax=max_abs
)

im = ax.imshow(
    differences,
    cmap="RdBu_r",
    norm=norm,
    interpolation="nearest"
)

for i in range(k):
    for j in range(k):

        if i == j:
            label = "—"
        else:
            label = (
                f"{differences[i, j]:+.2f}"
                f"{significance_stars(p_values[i, j])}"
            )

        color = (
            "white"
            if max_abs > 0 and
            abs(differences[i, j]) > 0.65 * max_abs
            else "#202020"
        )

        ax.text(
            j, i,
            label,
            ha="center",
            va="center",
            fontsize=12,
            fontweight="bold",
            color=color
        )

ax.set_xticks(range(k))
ax.set_xticklabels(
    LABELS,
    rotation=35,
    ha="right",
    fontsize=11
)

ax.set_yticks(range(k))
ax.set_yticklabels(
    LABELS,
    fontsize=11
)

ax.set_xlabel(
    "Column state",
    fontsize=12,
    fontweight="bold",
    labelpad=12
)

ax.set_ylabel(
    "Row state",
    fontsize=12,
    fontweight="bold",
    labelpad=12
)

# White borders between cells.
ax.set_xticks(
    np.arange(-0.5, k, 1),
    minor=True
)
ax.set_yticks(
    np.arange(-0.5, k, 1),
    minor=True
)

ax.grid(
    which="minor",
    color="white",
    linewidth=2
)

ax.tick_params(
    which="minor",
    bottom=False,
    left=False
)

cbar = fig.colorbar(
    im,
    ax=ax,
    fraction=0.045,
    pad=0.04
)

cbar.set_label(
    "Δ Firing rate (Hz)\nColumn − Row",
    fontsize=11
)

ax.set_title(
    f"WT | DG | Firing rate\n"
    f"Paired behavioral-state differences (n={n_mice} mice)",
    fontsize=15,
    fontweight="bold",
    pad=18
)

fig.text(
    0.5, 0.015,
    "Positive = higher firing rate in column state. "
    "Negative = lower firing rate.\n"
    "* p<0.05   ** p<0.01   *** p<0.001 "
    "(paired Wilcoxon, Holm-adjusted over 15 pairs)",
    ha="center",
    fontsize=10
)

fig.subplots_adjust(
    bottom=0.22,
    top=0.86
)

# ------------------------------------------------------------
# 5. SAVE
# ------------------------------------------------------------

output = FIGURES / "WT_DG_firing_rate_pairwise_heatmap"

fig.savefig(
    output.with_suffix(".png"),
    dpi=350,
    bbox_inches="tight"
)

fig.savefig(
    output.with_suffix(".pdf"),
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

results.to_csv(
    FIGURES / "WT_DG_firing_rate_pairwise_statistics.csv",
    index=False
)

print("\nSIGNIFICANT PAIRS (HOLM p < 0.05)")
display(
    results.loc[
        results["p_holm"] < 0.05
    ].sort_values("p_holm").round(5)
)

print("\nSaved to:", FIGURES)
